# Fundamentos de RDD en Spark

**Kevin Daniel Aguilar Rodriguez**<br>
Ingeniería en Datos e Inteligencia Artificial<br>
Universidad Politécnica de Querétaro<br>
Manejo Masivo de Datos · Noviembre 2026

Contenido del notebook:

1. Ejercicios prácticos 1 a 8
2. Proyecto: análisis de calificaciones
3. Pregunta de reflexión: RDD vs DataFrame
4. Anexo: experimento narrow vs wide (sección 6.3 de la guía)

## Preparación

Spark corre en modo local con todos los núcleos disponibles (`local[*]`). Desactivo la barra de
progreso de consola para que las salidas queden limpias al exportar a PDF.

In [1]:
import os
import shutil
import time
from operator import add
from pathlib import Path

from pyspark import StorageLevel
from pyspark.sql import SparkSession

# El notebook vive en notebooks/, los datos en ../datos
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATOS = RAIZ / "datos"
RESULTADOS = RAIZ / "resultados"
RESULTADOS.mkdir(exist_ok=True)

spark = (SparkSession.builder
         .appName("PracticaRDD")
         .master("local[*]")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("ERROR")

print("Spark", spark.version, "| master:", sc.master, "| paralelismo por defecto:", sc.defaultParallelism)

Spark 3.5.3 | master: local[*] | paralelismo por defecto: 12


In [2]:
def guardar_texto(rdd, nombre):
    # saveAsTextFile falla si la carpeta ya existe, así que la borro antes.
    # coalesce(1) junta todo en un solo part-00000 (sin shuffle) porque el resultado es pequeño.
    salida = RESULTADOS / nombre
    if salida.exists():
        shutil.rmtree(salida)
    rdd.coalesce(1).saveAsTextFile(salida.as_uri())
    return salida

## Ejercicio 1: Word Count básico

Cuenta cuántas veces aparece cada palabra en `["hola mundo", "hola spark", "mundo spark"]`.

Primero `flatMap` parte cada frase y aplana el resultado en una sola lista de palabras; después cada
palabra se convierte en el par `(palabra, 1)` y `reduceByKey` suma los unos de cada clave.

In [3]:
texto = ["hola mundo", "hola spark", "mundo spark"]

conteo = (sc.parallelize(texto)
            .flatMap(lambda frase: frase.split(" "))
            .map(lambda palabra: (palabra, 1))
            .reduceByKey(add))

for palabra, veces in sorted(conteo.collect()):
    print(f"{palabra:<6} {veces}")

hola   2
mundo  2
spark  2


## Ejercicio 2: Análisis de ventas

Datos: `[("Madrid", 100), ("Barcelona", 200), ("Madrid", 50)]`

1. Suma de ventas por ciudad.
2. Ciudades con ventas mayores a 100.
3. Orden de mayor a menor.

In [4]:
ventas = sc.parallelize([("Madrid", 100), ("Barcelona", 200), ("Madrid", 50)])

por_ciudad = ventas.reduceByKey(add)
print("1. Total por ciudad:   ", sorted(por_ciudad.collect()))

mayores_100 = por_ciudad.filter(lambda par: par[1] > 100)
print("2. Ventas > 100:       ", sorted(mayores_100.collect()))

ordenadas = mayores_100.sortBy(lambda par: par[1], ascending=False)
print("3. De mayor a menor:   ", ordenadas.collect())

1. Total por ciudad:    [('Barcelona', 200), ('Madrid', 150)]
2. Ventas > 100:        [('Barcelona', 200), ('Madrid', 150)]


3. De mayor a menor:    [('Barcelona', 200), ('Madrid', 150)]


El filtro se aplica sobre el total ya agregado. Madrid vende 100 y 50 por separado, pero suma 150,
así que pasa el corte; si se filtrara antes de agregar, el 50 se perdería y el total de Madrid
saldría en 100.

## Ejercicio 3: Filtrado y transformación

Precios en USD `[10, 25, 40, 5, 100, 15]`: quedarse con los que son ≥ 20 y convertirlos a EUR (× 0.9).

In [5]:
precios_usd = sc.parallelize([10, 25, 40, 5, 100, 15])

precios_eur = (precios_usd
               .filter(lambda p: p >= 20)
               .map(lambda p: round(p * 0.9, 2)))

print("USD >= 20:", precios_usd.filter(lambda p: p >= 20).collect())
print("En EUR:   ", precios_eur.collect())

USD >= 20: [25, 40, 100]


En EUR:    [22.5, 36.0, 90.0]


Filtro antes de convertir: así la multiplicación solo se hace sobre los tres precios que importan.
Las dos operaciones son narrow, por lo que Spark las ejecuta en la misma pasada sobre cada partición.

## Ejercicio 4: Procesamiento de CSV

Dividir, aplanar y contar los productos únicos de `["Leche,Huevos,Pan", "Manzanas,Peras", "Leche,Chocolate"]`.

In [6]:
lineas = sc.parallelize(["Leche,Huevos,Pan", "Manzanas,Peras", "Leche,Chocolate"])

productos = lineas.flatMap(lambda linea: linea.split(","))
unicos = productos.distinct()

print("Productos (aplanados):", productos.collect())
print("Total de productos:   ", productos.count())
print("Productos únicos:     ", sorted(unicos.collect()))
print("Cantidad de únicos:   ", unicos.count())

Productos (aplanados): ['Leche', 'Huevos', 'Pan', 'Manzanas', 'Peras', 'Leche', 'Chocolate']


Total de productos:    7


Productos únicos:      ['Chocolate', 'Huevos', 'Leche', 'Manzanas', 'Pan', 'Peras']


Cantidad de únicos:    6


In [7]:
# Extra: cuántas veces aparece cada producto (Leche es el único repetido)
frecuencia = productos.map(lambda p: (p, 1)).reduceByKey(add).sortBy(lambda par: (-par[1], par[0]))
frecuencia.collect()

[('Leche', 2),
 ('Chocolate', 1),
 ('Huevos', 1),
 ('Manzanas', 1),
 ('Pan', 1),
 ('Peras', 1)]

## Ejercicio 5: Análisis de logs

El archivo simulado `datos/accesos.log` tiene el formato `ip,hora,metodo,url`:

```
192.168.1.1,10:00,GET,/index
192.168.1.2,10:01,POST,/login
192.168.1.1,10:02,GET,/imagen.jpg
```

In [8]:
# 1. Cargar el archivo
logs = sc.textFile((DATOS / "accesos.log").as_uri())
print("Líneas leídas:", logs.count())

# 2. Peticiones por IP: map a (ip, 1) + reduceByKey
peticiones_por_ip = (logs
                     .map(lambda linea: (linea.split(",")[0], 1))
                     .reduceByKey(add))
print("Peticiones por IP:", sorted(peticiones_por_ip.collect()))

# 3. IPs con más de una petición
ips_frecuentes = peticiones_por_ip.filter(lambda par: par[1] > 1)
print("IPs con > 1 petición:", ips_frecuentes.collect())

# 4. Guardar en un archivo de texto
salida = guardar_texto(ips_frecuentes, "ej05_ips_frecuentes")
print("\nGuardado en:", salida.relative_to(RAIZ))
for archivo in sorted(salida.glob("part-*")):
    print(f"  {archivo.name}: {archivo.read_text().strip()}")

Líneas leídas: 3
Peticiones por IP: [('192.168.1.1', 2), ('192.168.1.2', 1)]
IPs con > 1 petición: [('192.168.1.1', 2)]



Guardado en: resultados/ej05_ips_frecuentes
  part-00000: ('192.168.1.1', 2)


## Ejercicio 6: Pipeline y Shuffle

Ventas: `("Madrid", 100), ("Barcelona", 200), ("Madrid", 50), ("Valencia", 50)`

1. Filtrar las ventas menores de 60.
2. Sumar las ventas por ciudad.
3. Ordenar por facturación, de mayor a menor.

Interpreto "filtrar las ventas menores de 60" como descartarlas: el filtro deja pasar solo las
ventas de 60 o más. Como Valencia solo tiene una venta de 50, desaparece del resultado.

In [9]:
ventas = sc.parallelize([("Madrid", 100), ("Barcelona", 200), ("Madrid", 50), ("Valencia", 50)], 2)

ranking = (ventas
           .filter(lambda par: par[1] >= 60)          # narrow
           .reduceByKey(add)                         # wide: shuffle por ciudad
           .sortBy(lambda par: par[1], ascending=False))  # wide: shuffle para ordenar

print(ranking.collect())

[('Barcelona', 200), ('Madrid', 100)]


`toDebugString()` muestra el linaje. Cada bloque con sangría distinta es un *stage*; los cortes
aparecen justo donde hay shuffle (`reduceByKey` y `sortBy`). El `filter` va pegado a la lectura en el
mismo stage porque es narrow, y además reduce lo que viaja por la red: filtrar antes del
`reduceByKey` es la forma barata de hacerlo.

In [10]:
print(ranking.toDebugString().decode())

(2) PythonRDD[67] at collect at /tmp/ipykernel_44121/1178091173.py:8 []
 |  MapPartitionsRDD[66] at mapPartitions at PythonRDD.scala:160 []
 |  ShuffledRDD[65] at partitionBy at <unknown>:0 []
 +-(2) PairwiseRDD[64] at sortBy at /tmp/ipykernel_44121/1178091173.py:6 []
    |  PythonRDD[63] at sortBy at /tmp/ipykernel_44121/1178091173.py:6 []
    |  MapPartitionsRDD[60] at mapPartitions at PythonRDD.scala:160 []
    |  ShuffledRDD[59] at partitionBy at <unknown>:0 []
    +-(2) PairwiseRDD[58] at reduceByKey at /tmp/ipykernel_44121/1178091173.py:5 []
       |  PythonRDD[57] at reduceByKey at /tmp/ipykernel_44121/1178091173.py:5 []
       |  ParallelCollectionRDD[56] at readRDDFromFile at PythonRDD.scala:289 []


## Ejercicio 7: Diferencia entre map y flatMap

RDD con las frases `["Hola mundo", "Apache Spark es guay"]`.

In [11]:
frases = sc.parallelize(["Hola mundo", "Apache Spark es guay"])

con_map = frases.map(lambda f: f.split(" "))
con_flatmap = frases.flatMap(lambda f: f.split(" "))

print("map     ->", con_map.collect(), "| elementos:", con_map.count())
print("flatMap ->", con_flatmap.collect(), "| elementos:", con_flatmap.count())

map     -> [['Hola', 'mundo'], ['Apache', 'Spark', 'es', 'guay']] | elementos: 2


flatMap -> ['Hola', 'mundo', 'Apache', 'Spark', 'es', 'guay'] | elementos: 6


**1. ¿Qué estructura obtengo con `map`?** Una lista de listas. `map` produce exactamente una salida
por cada entrada, así que dos frases dan dos elementos, y cada elemento es la lista de palabras de
su frase.

**2. ¿Qué cambia con `flatMap`?** Que las listas internas se deshacen y cada palabra pasa a ser un
elemento independiente del RDD: de 2 elementos se pasa a 6. Es lo que se necesita para contar
palabras, porque `reduceByKey` y `distinct` trabajan sobre elementos sueltos, no sobre listas.

## Ejercicio 8: Optimización

*Tienes un RDD que usas 5 veces en diferentes cálculos. ¿Qué debes hacer para optimizar?*

Persistirlo antes de usarlo. Sin persistencia, cada acción recorre el linaje desde el principio, así
que las 5 acciones repiten 5 veces la lectura y todas las transformaciones. Con `cache()` o
`persist()`, la primera acción calcula el RDD y guarda sus particiones; las otras cuatro leen de
ahí.

Para elegir el nivel: si cabe en RAM, `cache()` (que es `MEMORY_ONLY`) basta; si puede no caber, es
mejor `persist(StorageLevel.MEMORY_AND_DISK)`, para que las particiones que no entren se escriban a
disco en lugar de recalcularse. Al terminar hay que llamar a `unpersist()` para liberar memoria.

La prueba de abajo usa un RDD con una transformación deliberadamente costosa y le aplica cinco
acciones distintas, primero sin caché y luego con `MEMORY_AND_DISK`.

In [12]:
def transformacion_costosa(x):
    # Simula un parseo o cálculo pesado por registro
    acumulado = 0
    for i in range(400):
        acumulado += (x * i) % 7
    return acumulado


def cinco_calculos(rdd):
    return (rdd.count(),
            rdd.sum(),
            rdd.max(),
            rdd.filter(lambda v: v > 90).count(),
            rdd.map(lambda v: v % 10).distinct().count())


base = sc.parallelize(range(200_000), 8)

sin_cache = base.map(transformacion_costosa)
inicio = time.perf_counter()
r1 = cinco_calculos(sin_cache)
t_sin = time.perf_counter() - inicio

con_cache = base.map(transformacion_costosa).persist(StorageLevel.MEMORY_AND_DISK)
inicio = time.perf_counter()
r2 = cinco_calculos(con_cache)
t_con = time.perf_counter() - inicio

print("Resultados iguales:", r1 == r2, r1)
print(f"Sin persistir:          {t_sin:6.2f} s")
print(f"Con MEMORY_AND_DISK:    {t_con:6.2f} s")
print(f"Mejora:                 {t_sin / t_con:6.1f}x")
print("Nivel de almacenamiento:", con_cache.getStorageLevel())

con_cache.unpersist()
print("¿Sigue en caché tras unpersist()?", con_cache.is_cached)

Resultados iguales: True (200000, 205199316, 1197, 171428, 2)
Sin persistir:            3.08 s
Con MEMORY_AND_DISK:      1.01 s
Mejora:                    3.0x
Nivel de almacenamiento: Disk Memory Serialized 1x Replicated
¿Sigue en caché tras unpersist()? False


Las cifras exactas cambian de una ejecución a otra, pero la versión persistida siempre gana porque
solo paga la transformación costosa una vez. En la pestaña *Storage* de la Spark UI
(`localhost:4040`) se ve el RDD mientras está persistido.

El nivel aparece como *Serialized* aunque pedí `MEMORY_AND_DISK`: en PySpark los datos siempre se
guardan serializados con pickle, así que en Python no existe una versión "deserializada".

## Proyecto: análisis de calificaciones

**Escenario.** Soy el administrador de un instituto y tengo las notas de tres asignaturas en
archivos de texto separados con formato `nombre,nota`. Hay que unificarlos y sacar un informe
usando RDDs.

Uso los archivos completos del repositorio de datos del curso (`josepgarcia/datos`) y no solo las
tres líneas del enunciado. Antes de programar, al revisarlos encontré esto:

| Archivo | Registros | Detalles |
| --- | ---: | --- |
| `notas_mates.txt` | 19 | Una nota decimal (Ramon, 4.5) |
| `notas_fisica.txt` | 20 | **Rocio aparece dos veces** (5 y 7) |
| `notas_ingles.txt` | 18 | **Pedro no tiene nota** |

En los tres archivos falta el salto de línea al final y hay un nombre compuesto ("Jose Juan").

**Decisión sobre el duplicado:** las dos notas de Rocio en física se promedian (queda con 6). El
duplicado se detecta y se reporta antes de corregirlo, para que no pase desapercibido.

**Preguntas que resuelve el proyecto:**

1. Crear un RDD por asignatura con tuplas `(alumno, nota)` y uno unificado.
2. Nota más baja de cada alumno.
3. Nota media de cada alumno.
4. Cuántos alumnos suspenden cada asignatura (nota < 5).
5. Asignatura en la que suspende más gente.
6. Total de notables o sobresalientes por alumno (nota ≥ 7).
7. Qué alumno no se ha presentado a inglés.
8. A cuántas asignaturas se ha presentado cada alumno.
9. RDD con cada alumno y todas sus notas.

### Carga y limpieza

La función `parsear` valida cada línea. Las que no tengan exactamente un nombre y una nota numérica
entre 0 y 10 se descartan y se cuentan con un *accumulator*, igual que en la sección 9.1 de la guía.

In [13]:
ASIGNATURAS = ["mates", "fisica", "ingles"]
lineas_invalidas = sc.accumulator(0)


def parsear(linea):
    partes = linea.strip().split(",")
    if len(partes) != 2 or not partes[0].strip():
        lineas_invalidas.add(1)
        return None
    try:
        nota = float(partes[1])
    except ValueError:
        lineas_invalidas.add(1)
        return None
    if not 0 <= nota <= 10:
        lineas_invalidas.add(1)
        return None
    return (partes[0].strip(), nota)


crudos = {}
for asignatura in ASIGNATURAS:
    ruta = (DATOS / f"notas_{asignatura}.txt").as_uri()
    crudos[asignatura] = (sc.textFile(ruta)
                            .filter(lambda l: l.strip() != "")
                            .map(parsear)
                            .filter(lambda r: r is not None))

for asignatura, rdd in crudos.items():
    print(f"{asignatura:<7} {rdd.count():>3} registros válidos")
print("Líneas inválidas descartadas:", lineas_invalidas.value)

mates    19 registros válidos
fisica   20 registros válidos


ingles   18 registros válidos
Líneas inválidas descartadas: 0


Busco alumnos repetidos dentro de cada asignatura contando cuántas veces aparece cada nombre:

In [14]:
for asignatura, rdd in crudos.items():
    repetidos = (rdd.map(lambda r: (r[0], 1))
                    .reduceByKey(add)
                    .filter(lambda par: par[1] > 1)
                    .collect())
    if repetidos:
        notas = sorted(rdd.filter(lambda r, n=repetidos[0][0]: r[0] == n).values().collect())
        print(f"{asignatura}: {repetidos} -> notas {notas}")
    else:
        print(f"{asignatura}: sin duplicados")

mates: sin duplicados


fisica: [('Rocio', 2)] -> notas [5.0, 7.0]


ingles: sin duplicados


### 1. Un RDD por asignatura y uno unificado

Para resolver el duplicado uso `aggregateByKey` con el acumulador `(suma, contador)`, el mismo patrón
de la sección 7.4. En los alumnos sin repetir el promedio es su propia nota, así que el paso es
seguro para todos.

El RDD unificado `notas` tiene la forma `(alumno, (asignatura, nota))` y se reutiliza en casi todas
las preguntas siguientes, por eso lo persisto (ejercicio 8 aplicado).

In [15]:
def promediar_duplicados(rdd):
    return (rdd.aggregateByKey((0.0, 0),
                               lambda acc, nota: (acc[0] + nota, acc[1] + 1),
                               lambda a, b: (a[0] + b[0], a[1] + b[1]))
               .mapValues(lambda acc: acc[0] / acc[1]))


mates = promediar_duplicados(crudos["mates"])
fisica = promediar_duplicados(crudos["fisica"])
ingles = promediar_duplicados(crudos["ingles"])

print("Rocio en física tras promediar:", fisica.filter(lambda r: r[0] == "Rocio").collect())
print("mates:", mates.count(), "| fisica:", fisica.count(), "| ingles:", ingles.count())

notas = (mates.mapValues(lambda n: ("mates", n))
         .union(fisica.mapValues(lambda n: ("fisica", n)))
         .union(ingles.mapValues(lambda n: ("ingles", n)))
         .persist(StorageLevel.MEMORY_AND_DISK))

print("\nRegistros en el RDD unificado:", notas.count())
notas.take(5)

Rocio en física tras promediar: [('Rocio', 6.0)]


mates: 19 | fisica: 19 | ingles: 18



Registros en el RDD unificado: 56


[('Susana', ('mates', 9.0)),
 ('Pedro', ('mates', 5.0)),
 ('Rocio', ('mates', 6.0)),
 ('Triana', ('mates', 3.0)),
 ('Leonardo', ('mates', 1.0))]

In [16]:
def fmt(nota):
    # 6.0 -> "6", 4.5 -> "4.5"
    return f"{nota:g}"

### 2. Nota más baja de cada alumno

`reduceByKey(min)`: se pre-agrega en cada partición y solo viaja el mínimo parcial.

In [17]:
nota_minima = notas.mapValues(lambda v: v[1]).reduceByKey(min).sortByKey()

for alumno, minima in nota_minima.collect():
    print(f"{alumno:<10} {fmt(minima)}")

Alejandro  3
Anabel     2
Andres     4
Angel      4
Carlos     4
Fernando   5
Isabel     7
Jorge      5
Jose Juan  3
Leonardo   1
Maria      2
Nicolas    2
Oscar      3
Pedro      2
Ramon      4.5
Rocio      4
Rosa       6
Susana     2
Triana     3


### 3. Nota media de cada alumno

Otra vez `aggregateByKey` con `(suma, contador)`. `reduceByKey` no sirve directamente porque el
promedio no es asociativo: el promedio de promedios parciales no da el promedio total.

In [18]:
nota_media = (notas
              .aggregateByKey((0.0, 0),
                              lambda acc, v: (acc[0] + v[1], acc[1] + 1),
                              lambda a, b: (a[0] + b[0], a[1] + b[1]))
              .mapValues(lambda acc: round(acc[0] / acc[1], 2))
              .sortBy(lambda par: (-par[1], par[0])))

for alumno, media in nota_media.collect():
    print(f"{alumno:<10} {media:5.2f}")

Isabel      7.67
Rosa        7.67
Fernando    7.00
Jorge       6.67
Susana      6.67
Ramon       6.50
Angel       6.33
Anabel      5.67
Carlos      5.33
Rocio       5.33
Alejandro   5.00
Oscar       5.00
Andres      4.67
Nicolas     4.67
Jose Juan   3.67
Leonardo    3.67
Maria       3.67
Pedro       3.50
Triana      3.33


### 4. Alumnos que suspenden cada asignatura

In [19]:
suspensos = (notas
             .filter(lambda r: r[1][1] < 5)
             .map(lambda r: (r[1][0], 1))
             .reduceByKey(add))

for asignatura, total in sorted(suspensos.collect()):
    print(f"{asignatura:<7} {total} suspensos")

fisica  8 suspensos
ingles  7 suspensos
mates   7 suspensos


In [20]:
# Para ver quiénes son
(notas.filter(lambda r: r[1][1] < 5)
      .map(lambda r: (r[1][0], r[0]))
      .groupByKey()
      .mapValues(sorted)
      .sortByKey()
      .collect())

[('fisica',
  ['Alejandro',
   'Anabel',
   'Andres',
   'Carlos',
   'Jose Juan',
   'Maria',
   'Pedro',
   'Triana']),
 ('ingles',
  ['Angel', 'Jose Juan', 'Leonardo', 'Oscar', 'Rocio', 'Susana', 'Triana']),
 ('mates',
  ['Andres', 'Carlos', 'Leonardo', 'Maria', 'Nicolas', 'Ramon', 'Triana'])]

### 5. Asignatura con más suspensos

In [21]:
peor_asignatura, max_suspensos = suspensos.reduce(lambda a, b: a if a[1] >= b[1] else b)
print(f"La asignatura con más suspensos es {peor_asignatura}, con {max_suspensos}.")

La asignatura con más suspensos es fisica, con 8.


### 6. Notables o sobresalientes por alumno (nota ≥ 7)

Los alumnos que no tienen ninguna nota ≥ 7 no aparecerían con un simple `filter` + `reduceByKey`.
Para que salgan con 0 parto de la lista completa de alumnos y les sumo los conteos con
`leftOuterJoin`.

In [22]:
alumnos = notas.keys().distinct()

notables = (notas.filter(lambda r: r[1][1] >= 7)
                 .map(lambda r: (r[0], 1))
                 .reduceByKey(add))

notables_por_alumno = (alumnos.map(lambda a: (a, None))
                       .leftOuterJoin(notables)
                       .mapValues(lambda v: v[1] or 0)
                       .sortBy(lambda par: (-par[1], par[0])))

for alumno, total in notables_por_alumno.collect():
    print(f"{alumno:<10} {total}")

Isabel     3
Anabel     2
Fernando   2
Ramon      2
Rosa       2
Susana     2
Alejandro  1
Angel      1
Carlos     1
Jorge      1
Nicolas    1
Oscar      1
Andres     0
Jose Juan  0
Leonardo   0
Maria      0
Pedro      0
Rocio      0
Triana     0


### 7. Alumno que no se ha presentado a inglés

Operación de conjuntos: todos los alumnos menos los que aparecen en inglés (`subtract`).

In [23]:
sin_ingles = alumnos.subtract(ingles.keys())
print("No se presentó a inglés:", sin_ingles.collect())

No se presentó a inglés: ['Pedro']


### 8. Número de asignaturas a las que se presentó cada alumno

In [24]:
asignaturas_por_alumno = notas.map(lambda r: (r[0], 1)).reduceByKey(add).sortByKey()

print(asignaturas_por_alumno.collect())
print("\nAlumnos con menos de 3 asignaturas:",
      asignaturas_por_alumno.filter(lambda par: par[1] < 3).collect())

[('Alejandro', 3), ('Anabel', 3), ('Andres', 3), ('Angel', 3), ('Carlos', 3), ('Fernando', 3), ('Isabel', 3), ('Jorge', 3), ('Jose Juan', 3), ('Leonardo', 3), ('Maria', 3), ('Nicolas', 3), ('Oscar', 3), ('Pedro', 2), ('Ramon', 3), ('Rocio', 3), ('Rosa', 3), ('Susana', 3), ('Triana', 3)]

Alumnos con menos de 3 asignaturas: [('Pedro', 2)]


Rocio cuenta 3 y no 4 porque su duplicado de física ya se había resuelto en el paso 1. Sin esa
limpieza, este conteo habría salido mal.

### 9. RDD con cada alumno y sus notas

Aquí sí uso `groupByKey`: la pregunta pide literalmente *todas* las notas de cada alumno, que es el
único caso en que la guía lo considera adecuado. No hay nada que pre-agregar, así que
`reduceByKey` no ahorraría tráfico.

In [25]:
notas_por_alumno = notas.groupByKey().mapValues(dict).sortByKey()

print(f"{'Alumno':<10} {'Mates':>6} {'Física':>7} {'Inglés':>7}")
for alumno, por_asignatura in notas_por_alumno.collect():
    celdas = [fmt(por_asignatura[a]) if a in por_asignatura else "-" for a in ASIGNATURAS]
    print(f"{alumno:<10} {celdas[0]:>6} {celdas[1]:>7} {celdas[2]:>7}")

Alumno      Mates  Física  Inglés


Alejandro       5       3       7
Anabel          8       2       7
Andres          4       4       6
Angel           6       9       4
Carlos          4       4       8
Fernando        5       9       7
Isabel          8       8       7
Jorge          10       5       5
Jose Juan       5       3       3
Leonardo        1       6       4
Maria           2       3       6
Nicolas         2       7       5
Oscar           7       5       3
Pedro           5       2       -
Ramon         4.5       7       8
Rocio           6       6       4
Rosa            6       8       9
Susana          9       9       2
Triana          3       3       4


### Informe final

Junto todo en un solo RDD por alumno y lo guardo en `resultados/proyecto_informe` como CSV.

In [26]:
informe = (notas_por_alumno
           .join(nota_media)
           .join(nota_minima)
           .join(notables_por_alumno)
           .map(lambda r: (r[0], r[1][0][0][0], r[1][0][0][1], r[1][0][1], r[1][1]))
           .sortBy(lambda r: (-r[2], r[0])))


def a_csv(fila):
    alumno, por_asig, media, minima, notables = fila
    columnas = [fmt(por_asig[a]) if a in por_asig else "" for a in ASIGNATURAS]
    return ",".join([alumno, *columnas, f"{media:.2f}", fmt(minima), str(notables)])


encabezado = sc.parallelize(["alumno,mates,fisica,ingles,media,minima,notables"])
salida = guardar_texto(encabezado.union(informe.map(a_csv)), "proyecto_informe")
print("Guardado en:", salida.relative_to(RAIZ), "\n")
print((salida / "part-00000").read_text())

Guardado en: resultados/proyecto_informe 

alumno,mates,fisica,ingles,media,minima,notables
Isabel,8,8,7,7.67,7,3
Rosa,6,8,9,7.67,6,2
Fernando,5,9,7,7.00,5,2
Jorge,10,5,5,6.67,5,1
Susana,9,9,2,6.67,2,2
Ramon,4.5,7,8,6.50,4.5,2
Angel,6,9,4,6.33,4,1
Anabel,8,2,7,5.67,2,2
Carlos,4,4,8,5.33,4,1
Rocio,6,6,4,5.33,4,0
Alejandro,5,3,7,5.00,3,1
Oscar,7,5,3,5.00,3,1
Andres,4,4,6,4.67,4,0
Nicolas,2,7,5,4.67,2,1
Jose Juan,5,3,3,3.67,3,0
Leonardo,1,6,4,3.67,1,0
Maria,2,3,6,3.67,2,0
Pedro,5,2,,3.50,2,0
Triana,3,3,4,3.33,3,0



In [27]:
total_alumnos = alumnos.count()
aprobados_todo = nota_minima.filter(lambda par: par[1] >= 5).count()
media_max = nota_media.values().max()
media_min = nota_media.values().min()
mejores = nota_media.filter(lambda par: par[1] == media_max).keys().collect()
peores = nota_media.filter(lambda par: par[1] == media_min).keys().collect()

resumen = f'''Resumen del análisis
--------------------
Alumnos distintos:                 {total_alumnos}
Registros de notas (sin duplicar): {notas.count()}
Líneas inválidas descartadas:      {lineas_invalidas.value}
Duplicados corregidos:             Rocio en física (5 y 7 -> 6)
Aprueban todas sus asignaturas:    {aprobados_todo}
Mejor media:                       {" y ".join(mejores)} ({media_max:.2f})
Media más baja:                    {" y ".join(peores)} ({media_min:.2f})
Asignatura con más suspensos:      {peor_asignatura} ({max_suspensos})
No se presentó a inglés:           {", ".join(sin_ingles.collect())}
'''
print(resumen)
(RESULTADOS / "proyecto_resumen.txt").write_text(resumen, encoding="utf-8")
notas.unpersist();

Resumen del análisis
--------------------
Alumnos distintos:                 19
Registros de notas (sin duplicar): 56
Líneas inválidas descartadas:      0
Duplicados corregidos:             Rocio en física (5 y 7 -> 6)
Aprueban todas sus asignaturas:    4
Mejor media:                       Isabel y Rosa (7.67)
Media más baja:                    Triana (3.33)
Asignatura con más suspensos:      fisica (8)
No se presentó a inglés:           Pedro



## Pregunta de reflexión: ¿cuándo usaría un RDD en lugar de un DataFrame?

Usaría un RDD cuando los datos llegan sin una estructura confiable y la mayor parte del trabajo es
limpiarlos línea por línea antes de poder tratarlos como tabla. En el proyecto de calificaciones
los archivos traían un alumno duplicado, un alumno que faltaba en una materia y líneas sin salto
final; con `map`, `filter` y un accumulator pude validar cada registro y contar los inválidos con
mis propias reglas, mientras que un DataFrame me habría obligado a fijar un esquema desde la
lectura. Una vez limpio, el resultado sí lo pasaría a DataFrame para consultarlo con SQL y
aprovechar el optimizador Catalyst.

## Anexo: experimento narrow vs wide

Es el código de la sección 6.3 de la guía. Las dos pruebas recorren los mismos 5 millones de
números en 5 particiones, pero la segunda agrupa por el último dígito con `groupBy`, lo que obliga a
mover todos los datos entre particiones.

In [28]:
rdd_grande = sc.parallelize(range(5_000_000), 5)

inicio = time.perf_counter()
count_narrow = rdd_grande.map(lambda x: x * 2).filter(lambda x: x < 500).count()
t_narrow = time.perf_counter() - inicio

inicio = time.perf_counter()
count_wide = rdd_grande.groupBy(lambda x: x % 10).count()
t_wide = time.perf_counter() - inicio

print(f"Narrow (map + filter):  {t_narrow:6.2f} s  -> {count_narrow} elementos")
print(f"Wide (groupBy):         {t_wide:6.2f} s  -> {count_wide} grupos")
print(f"El wide tardó {t_wide / t_narrow:.1f} veces más")

Narrow (map + filter):    0.28 s  -> 250 elementos
Wide (groupBy):           0.64 s  -> 10 grupos
El wide tardó 2.2 veces más


En la pestaña *Stages* de la Spark UI, la prueba narrow aparece como un solo stage, mientras que la
wide se divide en dos: el primero escribe los datos del shuffle y el segundo los lee agrupados por
clave.

In [29]:
spark.stop()